# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Manish-Basnet-09/flyrank-ml-intership/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*





**Task type:** Scoring

My lane is CTR/Engagement Opportunity Scoring. The goal is to assign each content item an opportunity score based on measurable signals such as CTR, engagement, impressions, and search position. The score would help the content/SEO team prioritize which content items deserve further review. I am choosing scoring because the opportunity is likely to exist on a spectrum rather than being only a yes/no decision.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
task_type = "scoring"
lane = "CTR/Engagement Opportunity Scoring"

print("ML task type:", task_type)
print("Lane:", lane)

ML task type: scoring
Lane: CTR/Engagement Opportunity Scoring


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target/proxy:** The dataset does not contain a direct observed label showing whether a content item improved after a CTR or engagement intervention. I will therefore use trend_direction as a provisional observed proxy for performance opportunity. Content trending down may represent a stronger opportunity for investigation, while stable or upward-trending content may have lower immediate priority. The new category will be treated carefully because it may not have enough historical information. This proxy is a decision-support signal, not proof that an intervention will improve performance.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

print("\nTrend direction values:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nTrend direction by freshness tier:")
print(pd.crosstab(
    df["trend_direction"],
    df["freshness_tier"],
    margins=True
))

Dataset shape: (30000, 44)

Trend direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend direction by freshness tier:
freshness_tier    0-30  181+  31-90  91-180    All
trend_direction                                   
down             10473    82    103    5604  16262
flat               898    16      1     237   1152
new               2128    25      7      76   2236
stable            3813    24     26    2099   5962
up                3168    27     38    1155   4388
All              20480   174    175    9171  30000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric:** NDCG@K (Normalized Discounted Cumulative Gain).

 I would use NDCG@K to evaluate whether the scoring system places higher-opportunity content items near the top of the ranking. This matches the intended action because the content/SEO team will review a limited number of high-priority items rather than all 30,000 content items. A higher NDCG@K would indicate that the ranking is doing a better job of putting useful opportunities near the top. The value of K would depend on how many items the team can realistically review.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
metric = "NDCG@K"

print("Proposed success metric:", metric)
print("Higher value = better ranking quality at the top K items.")

Proposed success metric: NDCG@K
Higher value = better ranking quality at the top K items.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis:** One row represents one pseudonymized content item observed over the 90-day measurement window. The scoring decision will be made at the content-item level. Each item contains measurable signals such as CTR, engagement, impressions, search position, freshness, and trend information that can be used to assess its potential opportunity.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the starter dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Select columns relevant to the CTR/engagement opportunity lane
lane_df = df[
    [
        "content_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "engagement_rate",
        "avg_position",
        "content_age_days",
        "trend_direction"
    ]
]

print("Number of content items:", len(lane_df))

lane_df.head(10)

Number of content items: 30000


,content_id,impressions_90d,clicks_90d,ctr,engagement_rate,avg_position,content_age_days,trend_direction
0,content_304f48230142,3803,29,0.76,5.88,10.6,187,down
1,content_a1fb4e703a9e,15320,7,0.05,0.00,20.3,445,down
2,content_9aa793d4d895,12581,11,0.09,0.00,36.5,141,down
3,content_331d6c4de07b,11751,58,0.49,1.28,6.2,463,stable
4,content_d99b7a2d90ca,19140,24,0.13,0.00,44.0,263,down
5,content_d4084a4bc775,3970,1,0.03,0.00,8.5,147,down
6,content_9a34b442b552,20,0,0.00,0.00,7.0,90,down
7,content_a63219c6e95a,1724,1,0.06,3.57,21.2,445,stable
8,content_5e6c160719bc,32574,29,0.09,5.88,46.0,90,down
9,content_c27558df2b0c,1240,2,0.16,0.00,4.9,257,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple fixed rule such as "flag every content item with CTR below 0.2%" would not capture the different contexts in which a low CTR occurs. Content items vary substantially in impressions, search position, engagement, freshness, and trend direction. An ML-based scoring approach could combine these signals and learn relationships between them instead of relying on one manually chosen threshold. However, ML should only be considered better if it performs better than a simple rule-based baseline on an appropriate evaluation outcome.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simple fixed-rule baseline
baseline = df[
    [
        "ctr",
        "impressions_90d",
        "avg_position",
        "engagement_rate"
    ]
].copy()

baseline["low_ctr_rule"] = baseline["ctr"] < 0.2

print("Items flagged by the fixed CTR rule:")
print(baseline["low_ctr_rule"].value_counts())

baseline.head(10)

Items flagged by the fixed CTR rule:
low_ctr_rule
True     19983
False    10017
Name: count, dtype: int64


,ctr,impressions_90d,avg_position,engagement_rate,low_ctr_rule
0,0.76,3803,10.6,5.88,False
1,0.05,15320,20.3,0.00,True
2,0.09,12581,36.5,0.00,True
3,0.49,11751,6.2,1.28,False
4,0.13,19140,44.0,0.00,True
5,0.03,3970,8.5,0.00,True
6,0.00,20,7.0,0.00,True
7,0.06,1724,21.2,3.57,True
8,0.09,32574,46.0,5.88,True
9,0.16,1240,4.9,0.00,True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.